# Jackfruit run 1b: fixing the directions
Run 1 found two problems: (1) an **unrelated** scenario's direction worked almost as well as the real one, so the directions are mostly a shared "something unusual happened" signal; (2) edits near the answer may simply push the model towards **"Yes"**.

This notebook tests the fix on run 1's 17 valid scenarios, at the layers where edits worked (12, 16, 20):
* **Specific directions:** remove the shared component (the average direction over all 20 scenarios) from each direction, then rescale to the original length.
* **Yes-bias control:** three filler questions unrelated to any story. Their shift is subtracted from the cause and effect shifts.
* **Specificity test:** the real direction must raise its *own* mediator question clearly more than an unrelated scenario's direction does.

Reads run 1's `results/`. Runtime: T4, about 20 minutes. Use **Runtime → Run all**.

In [ ]:
# Mount Google Drive and work inside the jackfruit folder (story files live there; results are saved there)
import os
from google.colab import drive
drive.mount("/content/drive")
os.chdir("/content/drive/MyDrive/jackfruit")
OUT = "results"
os.makedirs(OUT, exist_ok=True)
print("working in", os.getcwd(), "| files:", sorted(os.listdir(".")))

In [ ]:
# ---------- Setup: model, tokenizer, settings
import gc, os, time, torch, numpy as np, pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
torch.set_grad_enabled(False)

NAME = "Qwen/Qwen3-4B-Base"
DEVICE = "cuda"
BS = 16                         # batch size; lower to 8 if you get out-of-memory errors
TRAIN_VARIANTS = range(0, 16)   # used only to learn directions
EVAL_VARIANTS = range(16, 20)   # held out; used for every score. Raise to range(16, 24) for the final run
LAYER_STEP = 4                  # plan of record: every 4th layer
ALPHAS = [1.0, 2.0, 4.0]        # edit strength, in units of the (true - false) difference vector
POSITIONS = ["mediator", "after", "answer"]

for v in ["lm", "model", "W32"]:
    if v in globals():
        del globals()[v]
gc.collect(); torch.cuda.empty_cache()

tok = AutoTokenizer.from_pretrained(NAME)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def load(dtype):
    m = AutoModelForCausalLM.from_pretrained(NAME, torch_dtype=dtype, device_map=DEVICE).eval()
    return m, m.get_output_embeddings().weight.float()     # float32 output layer

lm, W32 = load(torch.float16)
N_LAYERS = lm.config.num_hidden_layers
LAYERS = list(range(0, N_LAYERS, LAYER_STEP))
print("layers to sweep:", LAYERS)

YES = tok(" Yes", add_special_tokens=False).input_ids
NO = tok(" No", add_special_tokens=False).input_ids
assert len(YES) == 1 and len(NO) == 1
YES, NO = YES[0], NO[0]

In [ ]:
# ---------- Prompt building, token positions, belief with optional edit
import ast
FEWSHOT = (
    "Story: The kettle was on the stove and the water started to bubble.\n"
    "Question: Was the water hot?\nAnswer: Yes\n\n"
    "Story: Maya left her umbrella at home and the sky stayed clear all day.\n"
    "Question: Did Maya get wet from rain?\nAnswer: No\n\n"
)
STORY_OFFSET = len(FEWSHOT) + len("Story: ")

def prompt(story, q):
    return f"{FEWSHOT}Story: {story}\nQuestion: {q}\nAnswer:"

def encode(stories, qs, spans):
    """Tokenise a batch (left-padded) and find, per row, the last token of the key sentence."""
    enc = tok([prompt(s, q) for s, q in zip(stories, qs)], return_tensors="pt", padding=True,
              return_offsets_mapping=True)
    offs = enc.pop("offset_mapping")
    am = enc["attention_mask"]
    key_idx = []
    for i, (a, b) in enumerate(spans):
        a, b = a + STORY_OFFSET, b + STORY_OFFSET
        hit = ((offs[i, :, 0] < b) & (offs[i, :, 1] > a) & (am[i] == 1)).nonzero().flatten()
        assert len(hit) > 0, "key sentence not found in tokens"
        key_idx.append(hit[-1].item())
    return enc, torch.tensor(key_idx)

def position_mask(key_idx, T, mode):
    B = len(key_idx)
    m = torch.zeros(B, T)
    for i, k in enumerate(key_idx.tolist()):
        if mode == "mediator":
            m[i, k] = 1
        elif mode == "after":
            m[i, k + 1:T - 1] = 1     # tokens after the key sentence, excluding the final token
        elif mode == "answer":
            m[i, T - 1] = 1
    return m

def _layer_out(out):
    return out[0] if isinstance(out, tuple) else out

def belief(stories, qs, spans, edit=None, bs=BS):
    """log P(Yes) - log P(No). edit = dict(layer=int, vecs=[N, H] tensor, mode=str) or None."""
    res = []
    for i in range(0, len(stories), bs):
        enc, key_idx = encode(stories[i:i+bs], qs[i:i+bs], spans[i:i+bs])
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        handle = None
        if edit is not None:
            T = enc["input_ids"].shape[1]
            mask = position_mask(key_idx, T, edit["mode"]).to(DEVICE, lm.dtype)
            vecs = edit["vecs"][i:i+bs].to(DEVICE, lm.dtype)
            def hook(mod, inp, out, mask=mask, vecs=vecs):
                h = _layer_out(out) + mask[:, :, None] * vecs[:, None, :]
                return (h,) + tuple(out[1:]) if isinstance(out, tuple) else h
            handle = lm.model.layers[edit["layer"]].register_forward_hook(hook)
        try:
            h = lm.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
                         position_ids=pos).last_hidden_state[:, -1].float()
        finally:
            if handle is not None:
                handle.remove()
        logp = torch.log_softmax(h @ W32.T, dim=-1)
        res += (logp[:, YES] - logp[:, NO]).tolist()
    return np.array(res)

def key_states(stories, qs, spans, layers, bs=BS):
    """Residual stream (output of each layer in `layers`) at the last token of the key sentence."""
    out = []
    for i in range(0, len(stories), bs):
        enc, key_idx = encode(stories[i:i+bs], qs[i:i+bs], spans[i:i+bs])
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        hs = lm.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], position_ids=pos,
                      output_hidden_states=True).hidden_states
        rows = torch.arange(len(key_idx))
        out.append(torch.stack([hs[L + 1][rows, key_idx.to(DEVICE)].float().cpu() for L in layers], 1))
    return torch.cat(out)        # [N, len(layers), H]

# float16 overflow check; fall back to bfloat16 if needed
t = belief(["Priya ate a huge lunch and felt full."], ["Was Priya full?"], [[0, 5]])
if not np.isfinite(t).all():
    print("float16 overflow -> bfloat16")
    del lm, W32; gc.collect(); torch.cuda.empty_cache()
    lm, W32 = load(torch.bfloat16)
print("dtype:", lm.dtype)

d = pd.read_csv("stories_chain.csv")
d["variant"] = d.item_id.str.rsplit("_", n=1).str[1].astype(int)
for c in [c for c in d.columns if c.startswith("span_")]:
    d[c] = d[c].apply(ast.literal_eval)
train = d[d.variant.isin(TRAIN_VARIANTS)].reset_index(drop=True)
ev = d[d.variant.isin(EVAL_VARIANTS)].reset_index(drop=True)
SCEN = sorted(d.scenario_id.unique())
print(f"{len(SCEN)} scenarios | train items {len(train)} | eval items {len(ev)}")

In [ ]:
import glob, matplotlib.pyplot as plt
from scipy.stats import wilcoxon
sc = pd.read_csv(f"{OUT}/A_reference_scenarios.csv", index_col=0)
VALID = sc.index[sc.valid].tolist()
DIRS = torch.load(f"{OUT}/B_directions.pt")["dirs"]
FIX_LAYERS = [12, 16, 20]
LI = {L: LAYERS.index(L) for L in FIX_LAYERS}
FILLERS = ["Was there a cat nearby?", "Did someone in the story have a brother?", "Was a radio playing?"]
PROBES = {"cause": "q_cause", "effect": "q_eff", "med": "q_med"}
EV = ev[ev.scenario_id.isin(VALID)].reset_index(drop=True)
for i, f in enumerate(FILLERS):
    EV[f"q_f{i}"] = f
ALL_PROBES = {**PROBES, **{f"f{i}": f"q_f{i}" for i in range(len(FILLERS))}}

# shared component = mean direction over all 20 scenarios, per layer
M = torch.stack([DIRS[s] for s in SCEN]).mean(0)                      # [L, H]
Mhat = M / M.norm(dim=-1, keepdim=True)
def specific(d):
    p = d - (d * Mhat).sum(-1, keepdim=True) * Mhat
    return p / p.norm(dim=-1, keepdim=True) * d.norm(dim=-1, keepdim=True)
SPEC = {s: specific(DIRS[s]) for s in SCEN}
cos_shared = pd.DataFrame([dict(scenario_id=s, layer=L, cos_with_shared=torch.nn.functional.cosine_similarity(DIRS[s][LI[L]], M[LI[L]], dim=0).item())
                           for s in SCEN for L in FIX_LAYERS])
cos_shared.to_csv(f"{OUT}/X_cos_shared.csv", index=False)
display(cos_shared.groupby("layer").cos_with_shared.agg(["mean", "min", "max"]).round(2))
print("High cosine = most of each direction is the shared component.")

OTHER = {s: VALID[(VALID.index(s) + 1) % len(VALID)] for s in VALID}
def unrel(maps, s):
    o = maps[OTHER[s]]
    return o / o.norm(dim=-1, keepdim=True) * maps[s].norm(dim=-1, keepdim=True)
MAPS = {"raw": DIRS, "raw_unrelated": {s: unrel(DIRS, s) for s in VALID},
        "specific": SPEC, "specific_unrelated": {s: unrel(SPEC, s) for s in VALID}}

## Reference with the filler questions
The see and do *texts* could also shift the filler answers, so the reference gets the same correction.

In [ ]:
path = f"{OUT}/X_reference_items.csv"
if os.path.exists(path):
    xref = pd.read_csv(path)
else:
    xref = EV[["scenario_id", "item_id"]].copy()
    for cond in ["base", "see", "do"]:
        for p, q in ALL_PROBES.items():
            xref[f"{p}_{cond}"] = belief(EV[cond].tolist(), EV[q].tolist(), EV[f"span_{cond}"].tolist())
    xref.to_csv(path, index=False)
for cond in ["see", "do"]:
    for p in ALL_PROBES:
        xref[f"d{p}_{cond}"] = xref[f"{p}_{cond}"] - xref[f"{p}_base"]
    xref[f"dfill_{cond}"] = xref[[f"df{i}_{cond}" for i in range(len(FILLERS))]].mean(1)
xsc = xref.groupby("scenario_id")[[c for c in xref.columns if c.startswith("d")]].mean()
for cond in ["see", "do"]:
    xsc[f"dcauseC_{cond}"] = xsc[f"dcause_{cond}"] - xsc[f"dfill_{cond}"]
    xsc[f"deffectC_{cond}"] = xsc[f"deffect_{cond}"] - xsc[f"dfill_{cond}"]
xsc["r_see"] = xsc.dcauseC_see / xsc.deffectC_see
xsc["r_do"] = xsc.dcauseC_do / xsc.deffectC_do
xsc.to_csv(f"{OUT}/X_reference_scenarios.csv")
display(xsc[["dcause_see", "deffect_see", "dfill_see", "dcause_do", "deffect_do", "dfill_do", "r_see", "r_do"]].round(2))
print("Filler shift from the texts themselves (should be near 0):", xsc[["dfill_see", "dfill_do"]].mean().round(2).to_dict())

## The sweep
Layers 12, 16, 20 × positions mediator / after / answer. α = 4 for all four direction kinds (raw, raw-unrelated, specific, specific-unrelated); α = 2 also for raw and specific. Six questions per edit: cause, effect, mediator and the three fillers. Saved per layer.

In [ ]:
SETTINGS = [(k, 4.0) for k in MAPS] + [("raw", 2.0), ("specific", 2.0)]
stories, spans = EV.base.tolist(), EV.span_base.tolist()
base_b = {p: belief(stories, EV[q].tolist(), spans) for p, q in ALL_PROBES.items()}
print(f"{len(EV)} items x {len(FIX_LAYERS) * len(POSITIONS) * len(SETTINGS) * len(ALL_PROBES)} settings")
t0 = time.time()
for L in FIX_LAYERS:
    path = f"{OUT}/X_sweep_layer{L:02d}.csv"
    if os.path.exists(path):
        print("skip layer", L); continue
    rows = []
    for mode in POSITIONS:
        for kind, alpha in SETTINGS:
            vecs = torch.stack([alpha * MAPS[kind][s][LI[L]] for s in EV.scenario_id])
            r = EV[["scenario_id", "item_id"]].copy()
            r["layer"], r["position"], r["kind"], r["alpha"] = L, mode, kind, alpha
            for p, q in ALL_PROBES.items():
                r[f"d{p}"] = belief(stories, EV[q].tolist(), spans, edit=dict(layer=L, vecs=vecs, mode=mode)) - base_b[p]
            rows.append(r)
    pd.concat(rows).to_csv(path, index=False)
    print(f"layer {L} done ({(time.time() - t0) / 60:.1f} min)")

## Results
1. **Yes-bias:** how much do the filler questions move? If they move as much as the cause, the "seeing" result was just a yes-bias.
2. **Specificity:** real minus unrelated, on the edit's own mediator question (filler-corrected). A cell **passes** if the real direction beats the unrelated one by > 0.5 on average and in ≥ 70% of scenarios.
3. **Corrected D:** cause and effect shifts minus the filler shift, then the usual D, matched-effect rule and bootstrap CI.

In [ ]:
rng = np.random.default_rng(0)
def boot_median(x, n=2000):
    x = pd.Series(x).dropna().values
    if len(x) < 3:
        return np.nan, np.nan
    m = [np.median(rng.choice(x, len(x))) for _ in range(n)]
    return np.percentile(m, 2.5), np.percentile(m, 97.5)

sw = pd.concat(pd.read_csv(f) for f in sorted(glob.glob(f"{OUT}/X_sweep_layer*.csv")))
sw["dfill"] = sw[[f"df{i}" for i in range(len(FILLERS))]].mean(1)
for p in ["cause", "effect", "med"]:
    sw[f"d{p}C"] = sw[f"d{p}"] - sw.dfill
g = sw.groupby(["scenario_id", "layer", "position", "kind", "alpha"])[["dcause", "deffect", "dmed", "dfill", "dcauseC", "deffectC", "dmedC"]].mean().reset_index()

# 1. yes-bias
print("1. Mean shifts by direction kind (alpha = 4): cause, effect, mediator vs filler")
display(g[g.alpha == 4].groupby(["kind", "layer", "position"])[["dcause", "deffect", "dmed", "dfill"]].mean().round(2).unstack("kind"))

# 2. specificity on the mediator question
spec_rows = []
for base_kind in ["raw", "specific"]:
    a = g[(g.kind == base_kind) & (g.alpha == 4)].set_index(["scenario_id", "layer", "position"])
    b = g[(g.kind == f"{base_kind}_unrelated")].set_index(["scenario_id", "layer", "position"])
    diff = (a.dmedC - b.dmedC).reset_index(name="diff")
    for (L, pos), grp in diff.groupby(["layer", "position"]):
        p = wilcoxon(grp["diff"]).pvalue if (grp["diff"] != 0).sum() > 5 else np.nan
        spec_rows.append(dict(directions=base_kind, layer=L, position=pos, mean_real_minus_unrelated=grp["diff"].mean(),
                              share_real_wins=(grp["diff"] > 0).mean(), wilcoxon_p=p,
                              passes=(grp["diff"].mean() > 0.5) and ((grp["diff"] > 0).mean() >= 0.7)))
spec = pd.DataFrame(spec_rows); spec.to_csv(f"{OUT}/X_specificity.csv", index=False)
print("2. Specificity (mediator question, filler-corrected, alpha = 4)")
display(spec.round(3))

# 3. corrected D
g = g.merge(xsc[["deffectC_see", "r_see", "r_do"]], left_on="scenario_id", right_index=True)
g["works"] = g.deffectC >= 0.5 * g.deffectC_see
g["r_edit"] = g.dcauseC / g.deffectC
g["D"] = np.where(g.works, (g.r_see - g.r_edit) / (g.r_see - g.r_do), np.nan)
g["D_clip"] = g.D.clip(0, 1)
g.to_csv(f"{OUT}/X_scenario_scores.csv", index=False)
summ = []
for k, grp in g.groupby(["kind", "alpha", "layer", "position"]):
    lo, hi = boot_median(grp.D)
    summ.append(dict(zip(["kind", "alpha", "layer", "position"], k), works_rate=grp.works.mean(), n_scored=int(grp.D.notna().sum()),
                     D_median=grp.D.median(), D_clip_median=grp.D_clip.median(), ci_lo=lo, ci_hi=hi))
summ = pd.DataFrame(summ); summ.to_csv(f"{OUT}/X_summary.csv", index=False)
print("3. Corrected D, specific directions")
display(summ[summ.kind == "specific"].round(2))
print("Works rate by kind:"); display(summ.groupby(["kind", "alpha"]).works_rate.mean().round(3))
ok = spec[(spec.directions == "specific") & spec.passes]
print(f"\nSpecific directions pass the specificity test in {len(ok)} of {len(spec[spec.directions == 'specific'])} layer x position cells:",
      ok[["layer", "position"]].values.tolist())